# Module 5: LangSmith Engine — Close the Loop Automatically

> *"Your proactive agent engineer."* — LangSmith

> Part of the **Modular Workshops** series. Standalone, ~30 min. Runs great on its own, **as an alternative to Module 4**, or as a faster lap right after it.

In Module 4 you built the improvement loop **by hand**: trace → online eval scores a run → a run rule routes low scores to an annotation queue → a human reviews → fixes feed the next dataset.

**LangSmith Engine automates that entire loop** — and goes further. It reads your *deployed* agent's traces and runs five stages on its own:

1. **Detect** — find a *recurring* failure across many traces
2. **Diagnose** — trace the root cause into your connected GitHub source
3. **Propose a fix** — open a pull request (code or prompt)
4. **Deploy an evaluator** — add an online check so the issue can't silently regress
5. **Auto-reopen** — if the issue resurfaces after being resolved, reopen it

We run the **deployed `deep_agent` from Module 3** through an **assistant** (a saved configuration) called `engine-demo` that swaps its web-search tool for a subtly broken one — so Engine has a clear, recurring failure to find. Engine's analysis runs automatically in the background (~20 min for the first pass, then it rescans every ~6h), so a presenter primes it ahead of time; here we walk through the issue it found, the fix it proposes, and the evaluator it deploys.

<img src="../images/engine_issue_view.png" style="width: auto; max-height: 360px; border-radius: 8px;">

## Setup

Engine analyzes the **deployed** agent's traces. Deployed runs trace to the deployment's own project — **`modular-workshops-deep-agent`** — *not* your `.env` `LANGSMITH_PROJECT` (the platform reserves that var). We point issues, traces, datasets, and evaluators at that project.

The setup cell grabs the deployment URL straight from the `langgraph` CLI (no `.env` editing needed), then creates an **assistant** named `engine-demo` — a saved configuration of the `deep_agent` graph that pins `search_tool="easy"` (the deliberately broken search tool) with file-write interrupts off so seed runs finish unattended.

In [ ]:
import os
import re
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

from langsmith import Client
from langgraph_sdk import get_client

client = Client()

# Deployed runs trace to the deployment-named project -- LANGSMITH_PROJECT is reserved on the
# platform, so the .env value doesn't apply to the deployed agent. Engine analyzes THIS project.
DEPLOYMENT_NAME = "modular-workshops-deep-agent"
project_name = DEPLOYMENT_NAME

# Grab the deployment URL straight from the langgraph CLI -- no .env editing needed.
deployment_url = os.environ.get("LANGSMITH_DEPLOYMENT_URL")
rows = !langgraph deploy list --name-contains {DEPLOYMENT_NAME}
for line in rows:
    if re.search(rf"(^|\s){re.escape(DEPLOYMENT_NAME)}(\s|$)", line):
        m = re.search(r"https://\S+\.langgraph\.app", line)
        if m:
            deployment_url = m.group(0)
            break
if not deployment_url:
    raise RuntimeError(f"Couldn't find deployment {DEPLOYMENT_NAME!r}. Deploy it (Module 3) first.")
print("Deployment URL:", deployment_url)

sdk = get_client(url=deployment_url, api_key=os.environ.get("LANGSMITH_API_KEY"))

# Project deep link (where the deployed traces land).
try:
    print("Traces:", client.read_project(project_name=project_name).url)
except Exception as e:
    print(f"(project not created yet -- fine until the first seed run: {e})")

# Create or update the engine-demo assistant: deep_agent + the broken easy_search tool,
# the weak Context Hub guidance (context_repo), and file-write interrupts off so seed
# runs finish unattended.
desired_config = {"configurable": {
    "search_tool": "easy",
    "interrupts": False,
    "context_repo": "engine-demo-context",
}}
existing = await sdk.assistants.search(limit=100)
match = next(
    (a for a in existing if a.get("name") == "engine-demo" and a.get("graph_id") == "deep_agent"),
    None,
)
if match is None:
    engine_assistant = await sdk.assistants.create(
        "deep_agent", config=desired_config, name="engine-demo", if_exists="do_nothing",
    )
else:
    engine_assistant = await sdk.assistants.update(match["assistant_id"], config=desired_config)

assistant_id = engine_assistant["assistant_id"]
print("engine-demo assistant:", assistant_id)
print("  graph:", engine_assistant.get("graph_id"),
      "| config:", engine_assistant.get("config", {}).get("configurable"))

### Seed a few traces

This cell asks the `engine-demo` assistant a handful of research questions, giving Engine real traces to analyze (it needs only a couple to open an issue). The assistant runs its **full research workflow** — and because it uses `easy_search`, every web result comes back **without its content**, so the agent grounds on titles alone and produces vague, unsupported answers that recur across traces. (The assistant turns off the file-write HITL interrupt so these runs finish unattended.)

In [5]:
import asyncio

research_prompts = [
    "What is LangGraph used for?",
    "What does the LangChain v1 release add?",
    "What is LangSmith for?",
    "What is the difference between LangChain and LangGraph?",
    "What is Tavily?",
]

async def seed(q):
    await sdk.runs.wait(
        None,            # stateless run (no thread)
        assistant_id,    # the engine-demo assistant (deep_agent + easy_search, interrupts off)
        input={"messages": [{"role": "user", "content": q}]},
        metadata={"demo": "engine"},
    )
    print("seeded:", q)

# Fire all invocations concurrently so seeding finishes in one round-trip, not five.
await asyncio.gather(*(seed(q) for q in research_prompts))

print(f"\nSeeded {len(research_prompts)} traces into project: {project_name}")

seeded: What is Tavily?
seeded: What is LangSmith for?
seeded: What does the LangChain v1 release add?
seeded: What is LangGraph used for?
seeded: What is the difference between LangChain and LangGraph?

Seeded 5 traces into project: modular-workshops


## 1. The issue Engine filed

Open the **Engine** tab on your tracing project. Each issue Engine files has a **title**, a **severity**, a **category**, and a **description** (for us, something like *Research answers are ungrounded — search returns no content*). You can also list them from the CLI (the LangSmith SDK's issues endpoint isn't available in all workspaces yet):

In [ ]:
!langsmith project issues list --project "{project_name}"

## 2. Traces — how Engine uses traces

Engine starts from your **traces**. It clusters recurring failures, and every issue links the exact runs it learned from — open **Linked Traces** on the issue to inspect them. For `engine-demo`, those traces show `easy_search` returning titles and URLs but **no page content**, so the agent answers without real grounding.

Jump straight to the project's traces, and pull a few `easy_search` spans to see the missing content yourself:

In [ ]:
try:
    print("Inspect traces:", client.read_project(project_name=project_name).url, "\n")
except Exception as e:
    print(e)

search_runs = list(client.list_runs(
    project_name=project_name,
    filter='eq(name, "easy_search")',
    limit=3,
))
for r in search_runs:
    print("-", str(r.outputs)[:160])

## 3. Context — how Engine uses context

A trace tells Engine *what* broke; **context** tells it *why*. Engine reasons over everything you connect — your **GitHub source** *and* the agent's runtime context: its prompts, skills, and memory.

This assistant loads an `AGENTS.md` from the **Context Hub** repo `engine-demo-context`, seeded with deliberately weak guidance (*"lead with your own knowledge," "one search is enough," "don't cite sources," "fill in thin results from memory"*). Paired with the broken `easy_search` tool, that context nudges the agent toward confident, ungrounded answers. With the repo and context connected, Engine reads both and pins the root cause — the `easy_search` formatting that drops result content, and/or the instructions steering the agent away from grounding.

From the issue's **Proposed Fix**, **Engine writes the fix** and shows the diff with an explanation, then can **open a PR** (a code change *or* a prompt/context edit). Review it like a teammate's: open it, read the diff, decide — it's generated from *your* source, so let the proposal speak for itself.

## 4. Evaluations & datasets — how Engine uses evals

Engine turns the failing traces into **ground-truth examples** and can add them to a dataset, so the fix is backed by data you can re-run. Start from a small seed dataset; Engine's *Add offline examples* appends to it. The cell below creates (or reuses) that dataset and prints its link.

In [ ]:
dataset_name = "engine-demo-evals"
examples = [
    {"inputs": {"question": "What is LangGraph used for?"},
     "outputs": {"reference_answer": "A grounded explanation, citing real sources, that LangGraph orchestrates stateful, multi-step agent workflows."}},
    {"inputs": {"question": "What does the LangChain v1 release add?"},
     "outputs": {"reference_answer": "A grounded summary, citing sources, of create_agent and the middleware system."}},
    {"inputs": {"question": "What is LangSmith for?"},
     "outputs": {"reference_answer": "A grounded explanation, citing sources, of tracing, evals, and monitoring for LLM apps."}},
]

if client.has_dataset(dataset_name=dataset_name):
    ds = client.read_dataset(dataset_name=dataset_name)
else:
    ds = client.create_dataset(dataset_name)
    client.create_examples(
        inputs=[e["inputs"] for e in examples],
        outputs=[e["outputs"] for e in examples],
        dataset_id=ds.id,
    )

print(f"Dataset '{dataset_name}' ready -- Engine can add ground-truth examples here.")
print("View:", ds.url)

## 5. Online evaluators

A fix closes today's gap; an evaluator **prevents it from coming back**. Apply the issue's **Suggested Evaluator** in one click — it scores every new trace, and Engine reopens the issue if the regression returns. It's the same kind of online evaluator you'd build by hand in Module 4 with `create_run_rule(...)`; the optional cell below builds the equivalent check — *is the answer grounded in real sources?*

In [ ]:
from utils.langsmith_rules import create_run_rule

judge_prompt = (
    "You score whether the assistant's answer is grounded in real web content. "
    "Reply with grounded (true/false) and one sentence of comment. Mark false if the "
    "answer is vague, generic, or cites no substantive source content."
)
judge_schema = {
    "title": "grounded",
    "description": "Whether the answer is grounded in real source content.",
    "type": "object",
    "properties": {
        "grounded": {"type": "boolean", "description": "True if grounded in real source content"},
        "comment": {"type": "string", "description": "One short sentence explaining the score"},
    },
    "required": ["grounded", "comment"],
    "strict": True,
}

# Same helper as Module 4 -- the equivalent of the evaluator Engine suggests on the issue.
rule = create_run_rule(
    client,
    project_name=project_name,
    display_name="engine-regression-grounded",
    sampling_rate=1.0,
    filter="eq(is_root, true)",
    llm_judge_prompt=judge_prompt,
    llm_judge_schema=judge_schema,
)
print("Evaluator rule:", rule["id"])
print("Open in UI:", rule["url"])

## Close the loop (optional)

The main flow leaves the broken tool in place so the issue persists across demos. To watch Engine's loop close end to end:

1. **Merge** Engine's PR (it fixes `easy_search` so each result keeps its content).
2. **Redeploy** (`langgraph deploy` — same command from Module 3).
3. **Re-run the seed cell.** Answers are now grounded, the evaluator passes, and Engine **resolves** the issue on its next rescan.

To demo again, revert the fix and redeploy (the `engine-demo` assistant still points at `easy_search`) — Engine re-detects it.

*Good to know:* Engine keeps **running automatically in the background** — it rescans every ~6h and auto-reopens regressions on its own, can emit webhooks (issue detected / resolved / reopened), and is metered in LCUs (1 LCU = $1.50) with an Org-Admin spend limit.

## Recap — Engine vs. the hand-built loop

| Engine stage | You did this by hand in Module 4 | Engine automates |
|---|---|---|
| **Detect** | Queried traces with `list_runs` + filters | Finds *recurring* failures across traces on its own |
| **Diagnose** | Read traces, guessed the cause | Traces the cause into connected GitHub source |
| **Propose fix** | Edited code yourself | Opens a PR (code or prompt) |
| **Add an evaluator** | `create_run_rule(...)` | Suggests an evaluator you apply in one click |
| **Route to human** | Annotation queue rule | Files issues for review + recommends offline examples |
| **Re-check** | Re-ran evals manually | Rescans every ~6h, auto-reopens regressions |

Same loop you built in Module 4 — now detected, diagnosed, fixed, and guarded automatically.

**Docs:** [LangSmith Engine](https://docs.langchain.com/langsmith/engine) · [Assistants](https://docs.langchain.com/langsmith/assistants) · [Engine webhooks](https://docs.langchain.com/langsmith/engine-webhooks)